In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [29]:
!pip uninstall -y peft transformers torchao
!pip install -q \
transformers==4.42.4 \
peft==0.11.1 \
accelerate==0.31.0

Found existing installation: peft 0.11.1
Uninstalling peft-0.11.1:
  Successfully uninstalled peft-0.11.1
Found existing installation: transformers 4.42.4
Uninstalling transformers-4.42.4:
  Successfully uninstalled transformers-4.42.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


In [31]:
import transformers
import peft

print(transformers.__version__)
print(peft.__version__)

5.0.0
0.19.1


In [32]:
!pip install -q transformers==4.42.4 peft==0.11.1 accelerate==0.31.0

In [33]:
import transformers
import peft

print(transformers.__version__)
print(peft.__version__)

5.0.0
0.19.1


In [2]:
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
# q1
label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train["label"] = train["answer"].map(label_map)

print(train.loc[150,"label"])

2


In [6]:
#q2
row = train.iloc[0]

text = str(row["prompt"]) + " [SEP] " + str(row["B"])

print(text)

print(len(text))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
407


In [7]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
# q3
row = train.iloc[0]

choices = [
    str(row["prompt"])+" [SEP] "+str(row["A"]),
    str(row["prompt"])+" [SEP] "+str(row["B"]),
    str(row["prompt"])+" [SEP] "+str(row["C"]),
    str(row["prompt"])+" [SEP] "+str(row["D"]),
    str(row["prompt"])+" [SEP] "+str(row["E"])
]

enc = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = enc["input_ids"].unsqueeze(0)

print(input_ids.shape)

torch.Size([1, 5, 128])


In [9]:
# q4
all_input_ids = []

for i in range(16):

    row = train.iloc[i]

    choices = [
        row["prompt"]+" [SEP] "+row["A"],
        row["prompt"]+" [SEP] "+row["B"],
        row["prompt"]+" [SEP] "+row["C"],
        row["prompt"]+" [SEP] "+row["D"],
        row["prompt"]+" [SEP] "+row["E"]
    ]

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    all_input_ids.append(enc["input_ids"])

batch = torch.stack(all_input_ids)

print(batch.shape)

print(batch.numel())

torch.Size([16, 5, 128])
10240


In [10]:
# q5
model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

outputs = model(
    input_ids=input_ids,
    attention_mask=enc["attention_mask"].unsqueeze(0)
)

print(outputs.logits.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 5])


In [11]:
# q6
label = torch.tensor([train.loc[0,"label"]])

outputs = model(
    input_ids=input_ids,
    attention_mask=enc["attention_mask"].unsqueeze(0),
    labels=label
)

print(outputs.loss)

print(outputs.loss.shape)

print(outputs.loss.dim())

tensor(1.6126, grad_fn=<NllLossBackward0>)
torch.Size([])
0


In [34]:
# q7
from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, config)

model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(trainable)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693
295681


In [13]:
# q8
small = train.iloc[:100].copy()

data = []

for _,row in small.iterrows():

    choices = [
        row["prompt"]+" [SEP] "+row["A"],
        row["prompt"]+" [SEP] "+row["B"],
        row["prompt"]+" [SEP] "+row["C"],
        row["prompt"]+" [SEP] "+row["D"],
        row["prompt"]+" [SEP] "+row["E"]
    ]

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    data.append({
        "input_ids":enc["input_ids"],
        "attention_mask":enc["attention_mask"],
        "labels":row["label"]
    })

dataset = Dataset.from_list(data)

print(len(dataset))

print(torch.tensor(dataset[0]["input_ids"]).shape)

100
torch.Size([5, 128])


In [16]:
# q9
tiny = train.iloc[:32].copy()

data=[]

for _,row in tiny.iterrows():

    choices=[
        row["prompt"]+" [SEP] "+row["A"],
        row["prompt"]+" [SEP] "+row["B"],
        row["prompt"]+" [SEP] "+row["C"],
        row["prompt"]+" [SEP] "+row["D"],
        row["prompt"]+" [SEP] "+row["E"]
    ]

    enc=tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=64
    )

    data.append({
        "input_ids":enc["input_ids"],
        "attention_mask":enc["attention_mask"],
        "labels":row["label"]
    })

dataset=Dataset.from_list(data)

In [17]:
args=TrainingArguments(

    output_dir="out",

    per_device_train_batch_size=4,

    gradient_accumulation_steps=1,

    max_steps=4,

    logging_steps=1,

    report_to="none"

)

In [18]:
trainer=Trainer(
    model=model,
    args=args,
    train_dataset=dataset
)

trainer.train()

print(trainer.state.global_step)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
1,3.316156
2,3.151305
3,3.244152
4,3.209810


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

4


In [35]:
# q10
row=train.iloc[0]

choices=[
    row["prompt"]+" [SEP] "+row["A"],
    row["prompt"]+" [SEP] "+row["B"],
    row["prompt"]+" [SEP] "+row["C"],
    row["prompt"]+" [SEP] "+row["D"],
    row["prompt"]+" [SEP] "+row["E"]
]

enc=tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

with torch.no_grad():

    out=model(
        input_ids=enc["input_ids"].unsqueeze(0),
        attention_mask=enc["attention_mask"].unsqueeze(0)
    )

prob=torch.softmax(out.logits,dim=-1)

print(prob)

print(prob[0,4].item())

tensor([[0.1970, 0.1835, 0.1958, 0.2080, 0.2157]])
0.21573394536972046
